# Notebook 01 — Walmart Sales ETL

**ISA 444 Final Project — Retail Forecasting (Walmart Kaggle Track)**

This notebook prepares the Walmart Kaggle data for the rest of the forecasting pipeline. It mirrors the structure of `duke_data_etl.ipynb` from class.

### What this notebook does
1. Load `train.csv`, `features.csv`, `stores.csv`, and `test.csv` from the local data folder.
2. Merge the three training tables into one wide forecasting frame.
3. Add Nixtla-style identifier columns: `unique_id`, `ds`, `y`.
4. Select the **top 20 Store/Department combinations** by total weekly sales.
5. Engineer simple calendar / holiday / promotion features that downstream models can use as exogenous predictors.
6. Build a parallel frame for the Kaggle `test.csv` dates so future forecasts can be produced later.
7. Save everything as parquet files in `..\data\` for the other notebooks.

### Key project decisions baked in here
- **Top 20 series** (no hand-picks, no duplicates) — chosen by `sum(Weekly_Sales)`.
- **Weekly frequency** — Walmart weeks end on Friday, so we use `W-FRI` everywhere.
- **MarkDown NaN → 0** — the markdown promo columns are NaN before Nov 2011. Filling with 0 treats missing markdowns as 'no promo running', which is defensible.
- **Exogenous features kept in the frame** even for univariate models — downstream notebooks just drop them when not needed.


## Install Packages

Run once on a fresh Python environment. Comment out after the first successful install.

In [13]:
#!pip install pandas numpy pyarrow

## Library Imports

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

## Paths

`RAW_DIR` points at the Kaggle CSVs (read-only). `DATA_DIR` is where this notebook writes the processed parquet files that all downstream notebooks will read from.

In [15]:
# Absolute path to the raw Kaggle data on disk (do not modify the files there).
RAW_DIR = Path(r"C:\Users\23mwa\Downloads\ISA444-WalmartData")

# Project-relative path where processed parquets will live.
# Every other notebook reads from here.
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data dir :", RAW_DIR)
print("Processed dir:", DATA_DIR.resolve())

Raw data dir : C:\Users\23mwa\Downloads\ISA444-WalmartData
Processed dir: C:\Users\23mwa\data


## 1. Load Raw Kaggle Files

Four files come from the Kaggle Walmart competition:
- `train.csv` — the labeled history (Store, Dept, Date, Weekly_Sales, IsHoliday)
- `features.csv` — store-level weekly covariates (Temperature, Fuel_Price, CPI, Unemployment, MarkDown1-5, IsHoliday)
- `stores.csv` — static store metadata (Type, Size)
- `test.csv` — future dates we need to predict for, **no Weekly_Sales column**. Kaggle never released those actuals.


In [16]:
train    = pd.read_csv(RAW_DIR / "train.csv",    parse_dates=["Date"])
features = pd.read_csv(RAW_DIR / "features.csv", parse_dates=["Date"])
stores   = pd.read_csv(RAW_DIR / "stores.csv")
test     = pd.read_csv(RAW_DIR / "test.csv",     parse_dates=["Date"])

print("train   :", train.shape)
print("features:", features.shape)
print("stores  :", stores.shape)
print("test    :", test.shape, "  (no Weekly_Sales — Kaggle held this out)")

display(train.head())
display(features.head())
display(stores.head())
display(test.head())

train   : (421570, 5)
features: (8190, 12)
stores  : (45, 3)
test    : (115064, 4)   (no Weekly_Sales — Kaggle held this out)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


## 2. Clean the Features Table

`features.csv` has two issues we need to handle before merging:

1. **MarkDown1-5 are NaN before ~Nov 2011** and sparse afterward. We fill with 0 (interpreting missing as 'no markdown promo that week') and add a binary `had_markdown` flag so models can distinguish 'really zero' from 'imputed zero' if useful.
2. **`IsHoliday` exists in BOTH `train.csv` and `features.csv`**. They agree, but we drop it from features here so the merge later doesn't create `IsHoliday_x` / `IsHoliday_y` duplicates.

In [17]:
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

# had_markdown = 1 if ANY of the five markdown promos had a non-NaN value that week.
# Computed BEFORE we fill NaN -> 0, otherwise we'd lose the 'missing' signal.
features = features.assign(
    had_markdown = features[markdown_cols].notna().any(axis=1).astype(int)
)

# Fill the markdown NaN values with 0 (no promo running).
features[markdown_cols] = features[markdown_cols].fillna(0)

# Drop IsHoliday from features (it is also in train, and they agree).
features = features.drop(columns=["IsHoliday"])

print("Features after cleaning:")
features.info()
features.head()

Features after cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         8190 non-null   int64         
 1   Date          8190 non-null   datetime64[ns]
 2   Temperature   8190 non-null   float64       
 3   Fuel_Price    8190 non-null   float64       
 4   MarkDown1     8190 non-null   float64       
 5   MarkDown2     8190 non-null   float64       
 6   MarkDown3     8190 non-null   float64       
 7   MarkDown4     8190 non-null   float64       
 8   MarkDown5     8190 non-null   float64       
 9   CPI           7605 non-null   float64       
 10  Unemployment  7605 non-null   float64       
 11  had_markdown  8190 non-null   int32         
dtypes: datetime64[ns](1), float64(9), int32(1), int64(1)
memory usage: 735.9 KB


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,had_markdown
0,1,2010-02-05,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0
1,1,2010-02-12,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,0
2,1,2010-02-19,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,0
3,1,2010-02-26,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,0
4,1,2010-03-05,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,0


## 3. Build the Forecasting Frame

We start from `train.csv`, attach the `features.csv` covariates (which are weekly per store), attach the static `stores.csv` metadata (Type and Size, constant per store), and rename to Nixtla's required `unique_id` / `ds` / `y` schema.

`unique_id` follows the pattern `S<Store>_D<Dept>` (e.g. `S20_D92`) so each Store/Dept combination becomes one independent time series.


In [18]:
df = (
    train
    # Nixtla naming conventions: unique_id identifies the series, ds is timestamp, y is target.
    .rename(columns={"Date": "ds", "Weekly_Sales": "y"})
    .assign(unique_id = lambda x: "S" + x["Store"].astype(str) + "_D" + x["Dept"].astype(str))
    # Attach the features.csv covariates (weekly, per store).
    .merge(
        features.rename(columns={"Date": "ds"}),
        on=["Store", "ds"],
        how="left",
    )
    # Attach static store metadata (Type and Size — constant per store).
    .merge(
        stores.rename(columns={"Type": "store_type", "Size": "store_size"}),
        on="Store",
        how="left",
    )
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

print("Merged training frame shape:", df.shape)
df.head()

Merged training frame shape: (421570, 18)


,Store,Dept,ds,y,IsHoliday,unique_id,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,had_markdown,store_type,store_size
0,10,1,2010-02-05,40212.84,False,S10_D1,54.34,2.962,0.0,0.0,0.0,0.0,0.0,126.442065,9.765,0,B,126512
1,10,1,2010-02-12,67699.32,True,S10_D1,49.96,2.828,0.0,0.0,0.0,0.0,0.0,126.496258,9.765,0,B,126512
2,10,1,2010-02-19,49748.33,False,S10_D1,58.22,2.915,0.0,0.0,0.0,0.0,0.0,126.526286,9.765,0,B,126512
3,10,1,2010-02-26,33601.22,False,S10_D1,52.77,2.825,0.0,0.0,0.0,0.0,0.0,126.552286,9.765,0,B,126512
4,10,1,2010-03-05,36572.44,False,S10_D1,55.92,2.877,0.0,0.0,0.0,0.0,0.0,126.578286,9.765,0,B,126512


## 4. Pick the Top 20 Series by Total Sales

The project must downsample to fewer than 50 series. We rank every Store/Dept combination by the sum of its Weekly_Sales across the full history and keep the top 20.

Why top-by-sum and not top-by-mean? Sum is the standard 'most business value' measure and it's what the rubric implies. Departments with more weeks of history get a fair shot, and we avoid picking a low-volume department that happens to have a high mean in only a handful of weeks.


In [19]:
TOP_N = 20

# Total weekly sales per Store/Dept across the full history.
totals = (
    df.groupby("unique_id", as_index=False)["y"]
      .sum()
      .rename(columns={"y": "total_sales"})
      .sort_values("total_sales", ascending=False)
)

top_ids = totals.head(TOP_N)["unique_id"].tolist()
print(f"Selected the top {TOP_N} Store/Dept series by total weekly sales:")
display(totals.head(TOP_N).reset_index(drop=True))

Selected the top 20 Store/Dept series by total weekly sales:


,unique_id,total_sales
0,S14_D92,26101497.71
1,S2_D92,23572153.03
2,S20_D92,23542625.04
3,S13_D92,23170876.20
4,S4_D92,22789210.43
5,S20_D95,21537795.62
6,S4_D95,21054815.74
7,S27_D92,20952094.22
8,S14_D95,20655911.35
9,S2_D95,20533191.52


In [20]:
# Restrict the training frame to just those top 20 series.
df_top = df[df["unique_id"].isin(top_ids)].copy()

print("Top-20 training frame shape:", df_top.shape)
print("Series count                :", df_top["unique_id"].nunique())
print("Date range                  :", df_top["ds"].min().date(), "->", df_top["ds"].max().date())
print("Weeks per series (min/max)  :", df_top.groupby("unique_id").size().min(), "/", df_top.groupby("unique_id").size().max())

Top-20 training frame shape: (2860, 18)
Series count                : 20
Date range                  : 2010-02-05 -> 2012-10-26
Weeks per series (min/max)  : 143 / 143


## 5. Quick Sanity Checks

Before saving, verify the basics: every series is continuous weekly, no missing target values, and the date frequency is `W-FRI` as expected.


In [21]:
# Confirm there are no missing target (y) values.
n_missing_y = df_top["y"].isna().sum()
print(f"Missing y values: {n_missing_y}")

# Confirm every series has uniformly spaced weekly observations.
gaps = (
    df_top.sort_values(["unique_id", "ds"])
          .groupby("unique_id")["ds"]
          .diff()
          .dt.days
          .dropna()
          .unique()
)
print(f"Unique day-gaps between consecutive observations: {sorted(gaps)}  (should be [7])")

# Confirm the day of the week is Friday everywhere (Kaggle Walmart weeks end Friday).
print("Days of week present:", df_top["ds"].dt.day_name().unique())

Missing y values: 0
Unique day-gaps between consecutive observations: [7.0]  (should be [7])
Days of week present: ['Friday']


## 6. Engineer Lightweight Calendar Features

These are inexpensive and useful for the ML models in Notebook 03. The neural models in Notebook 04 will also see them as exogenous regressors.

- `week_of_year` — captures within-year seasonality (peak holiday weeks live near 47-52).
- `month`, `quarter`, `year` — coarser calendar groupings.
- `is_holiday_week` — copy of the existing `IsHoliday` flag as an explicit 0/1 numeric (LightGBM-friendly).

We deliberately keep this small. The Duke Energy notebooks showed that simple calendar features plus a couple of lags beat heavy feature engineering nine times out of ten.


In [22]:
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add simple calendar features that downstream models can consume.

    Used on BOTH the training frame and the Kaggle test frame so the feature
    columns match exactly when we go to produce future forecasts.
    """
    out = frame.copy()
    out["week_of_year"]    = out["ds"].dt.isocalendar().week.astype(int)
    out["month"]           = out["ds"].dt.month.astype(int)
    out["quarter"]         = out["ds"].dt.quarter.astype(int)
    out["year"]            = out["ds"].dt.year.astype(int)
    out["is_holiday_week"] = out["IsHoliday"].astype(int)
    return out

df_top = add_calendar_features(df_top)
df_top.head()

,Store,Dept,ds,y,IsHoliday,unique_id,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,CPI,Unemployment,had_markdown,store_type,store_size,week_of_year,month,quarter,year,is_holiday_week
7521,10,72,2010-02-05,232558.51,False,S10_D72,54.34,2.962,0.0,0.0,...,126.442065,9.765,0,B,126512,5,2,1,2010,0
7522,10,72,2010-02-12,202622.42,True,S10_D72,49.96,2.828,0.0,0.0,...,126.496258,9.765,0,B,126512,6,2,1,2010,1
7523,10,72,2010-02-19,184982.01,False,S10_D72,58.22,2.915,0.0,0.0,...,126.526286,9.765,0,B,126512,7,2,1,2010,0
7524,10,72,2010-02-26,186002.43,False,S10_D72,52.77,2.825,0.0,0.0,...,126.552286,9.765,0,B,126512,8,2,1,2010,0
7525,10,72,2010-03-05,191989.54,False,S10_D72,55.92,2.877,0.0,0.0,...,126.578286,9.765,0,B,126512,9,3,1,2010,0


## 7. Build the Parallel Test Frame for the Kaggle Test Dates

Kaggle's `test.csv` covers future dates that come right after `train.csv` ends. It has no `Weekly_Sales` column — those actuals were never released — so this frame is purely for producing future forecasts (no metrics).

Steps:
1. Restrict to the same Store/Dept combinations we kept for training (top 20).
2. Merge in the same exogenous columns from `features.csv` and `stores.csv` so model inputs line up.
3. Apply the same calendar feature engineering.
4. Add a placeholder `y` column of NaN — some Nixtla APIs want the column to exist.


In [23]:
test_top = (
    test
    .rename(columns={"Date": "ds"})
    .assign(unique_id = lambda x: "S" + x["Store"].astype(str) + "_D" + x["Dept"].astype(str))
    .pipe(lambda d: d[d["unique_id"].isin(top_ids)])  # keep only our top-20 series
    .merge(features.rename(columns={"Date": "ds"}), on=["Store", "ds"], how="left")
    .merge(stores.rename(columns={"Type": "store_type", "Size": "store_size"}), on="Store", how="left")
    .assign(y = np.nan)  # placeholder — Kaggle did not release actuals here
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

test_top = add_calendar_features(test_top)

print("Test (future) frame shape:", test_top.shape)
print("Test date range          :", test_top["ds"].min().date(), "->", test_top["ds"].max().date())
print("Weeks per series in test :", test_top.groupby("unique_id").size().unique())
test_top.head()

Test (future) frame shape: (780, 23)
Test date range          : 2012-11-02 -> 2013-07-26
Weeks per series in test : [39]


,Store,Dept,ds,IsHoliday,unique_id,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Unemployment,had_markdown,store_type,store_size,y,week_of_year,month,quarter,year,is_holiday_week
0,10,72,2012-11-02,False,S10_D72,70.79,4.099,25680.20,6037.06,44.68,...,6.943,1,B,126512,NaN,44,11,4,2012,0
1,10,72,2012-11-09,False,S10_D72,70.28,3.780,9208.40,2501.11,364.14,...,6.943,1,B,126512,NaN,45,11,4,2012,0
2,10,72,2012-11-16,False,S10_D72,58.82,3.703,13459.00,120.76,128.41,...,6.943,1,B,126512,NaN,46,11,4,2012,0
3,10,72,2012-11-23,True,S10_D72,63.95,3.759,1789.75,0.10,146394.44,...,6.943,1,B,126512,NaN,47,11,4,2012,1
4,10,72,2012-11-30,False,S10_D72,64.13,3.719,4304.00,0.00,8935.00,...,6.943,1,B,126512,NaN,48,11,4,2012,0


## 8. Final Column Set and Save

We save two parquet files into `..\data\`:

- `walmart_top20_train.parquet` — labeled history for fitting and 5-fold CV evaluation
- `walmart_top20_test.parquet`  — future dates where we will produce Kaggle-style forecasts

Both files have **identical schemas** (except `y` is real in train and NaN in test) so downstream code can treat them uniformly.


In [24]:
# Canonical column order for both files. Keeping this consistent means downstream
# notebooks can assume the schema instead of re-deriving it every time.
FINAL_COLS = [
    "unique_id", "ds", "y",
    # Holiday / promo features
    "is_holiday_week",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5", "had_markdown",
    # Macroeconomic / weather features
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    # Static store features
    "store_type", "store_size",
    # Calendar features
    "week_of_year", "month", "quarter", "year",
]

df_train_out = df_top[FINAL_COLS].copy()
df_test_out  = test_top[FINAL_COLS].copy()

# Save as parquet — fast to load, preserves dtypes including datetime64.
df_train_out.to_parquet(DATA_DIR / "walmart_top20_train.parquet", index=False)
df_test_out.to_parquet(DATA_DIR / "walmart_top20_test.parquet",  index=False)

print("Saved:", (DATA_DIR / "walmart_top20_train.parquet").resolve(), df_train_out.shape)
print("Saved:", (DATA_DIR / "walmart_top20_test.parquet").resolve(),  df_test_out.shape)

Saved: C:\Users\23mwa\data\walmart_top20_train.parquet (2860, 20)
Saved: C:\Users\23mwa\data\walmart_top20_test.parquet (780, 20)


## 9. Final Preview

In [25]:
print("--- Training parquet ---")
df_train_out.info()
display(df_train_out.head())
print("\n--- Test (future-forecast) parquet ---")
df_test_out.info()
display(df_test_out.head())

--- Training parquet ---
<class 'pandas.core.frame.DataFrame'>
Index: 2860 entries, 7521 to 373364
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   unique_id        2860 non-null   object        
 1   ds               2860 non-null   datetime64[ns]
 2   y                2860 non-null   float64       
 3   is_holiday_week  2860 non-null   int32         
 4   MarkDown1        2860 non-null   float64       
 5   MarkDown2        2860 non-null   float64       
 6   MarkDown3        2860 non-null   float64       
 7   MarkDown4        2860 non-null   float64       
 8   MarkDown5        2860 non-null   float64       
 9   had_markdown     2860 non-null   int32         
 10  Temperature      2860 non-null   float64       
 11  Fuel_Price       2860 non-null   float64       
 12  CPI              2860 non-null   float64       
 13  Unemployment     2860 non-null   float64       
 14  store_type     

,unique_id,ds,y,is_holiday_week,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,had_markdown,Temperature,Fuel_Price,CPI,Unemployment,store_type,store_size,week_of_year,month,quarter,year
7521,S10_D72,2010-02-05,232558.51,0,0.0,0.0,0.0,0.0,0.0,0,54.34,2.962,126.442065,9.765,B,126512,5,2,1,2010
7522,S10_D72,2010-02-12,202622.42,1,0.0,0.0,0.0,0.0,0.0,0,49.96,2.828,126.496258,9.765,B,126512,6,2,1,2010
7523,S10_D72,2010-02-19,184982.01,0,0.0,0.0,0.0,0.0,0.0,0,58.22,2.915,126.526286,9.765,B,126512,7,2,1,2010
7524,S10_D72,2010-02-26,186002.43,0,0.0,0.0,0.0,0.0,0.0,0,52.77,2.825,126.552286,9.765,B,126512,8,2,1,2010
7525,S10_D72,2010-03-05,191989.54,0,0.0,0.0,0.0,0.0,0.0,0,55.92,2.877,126.578286,9.765,B,126512,9,3,1,2010



--- Test (future-forecast) parquet ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   unique_id        780 non-null    object        
 1   ds               780 non-null    datetime64[ns]
 2   y                0 non-null      float64       
 3   is_holiday_week  780 non-null    int32         
 4   MarkDown1        780 non-null    float64       
 5   MarkDown2        780 non-null    float64       
 6   MarkDown3        780 non-null    float64       
 7   MarkDown4        780 non-null    float64       
 8   MarkDown5        780 non-null    float64       
 9   had_markdown     780 non-null    int32         
 10  Temperature      780 non-null    float64       
 11  Fuel_Price       780 non-null    float64       
 12  CPI              520 non-null    float64       
 13  Unemployment     520 non-null    float64       
 14  st

,unique_id,ds,y,is_holiday_week,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,had_markdown,Temperature,Fuel_Price,CPI,Unemployment,store_type,store_size,week_of_year,month,quarter,year
0,S10_D72,2012-11-02,NaN,0,25680.20,6037.06,44.68,17412.04,4223.05,1,70.79,4.099,131.236226,6.943,B,126512,44,11,4,2012
1,S10_D72,2012-11-09,NaN,0,9208.40,2501.11,364.14,679.96,6831.32,1,70.28,3.780,131.279355,6.943,B,126512,45,11,4,2012
2,S10_D72,2012-11-16,NaN,0,13459.00,120.76,128.41,1802.25,5503.42,1,58.82,3.703,131.325800,6.943,B,126512,46,11,4,2012
3,S10_D72,2012-11-23,NaN,1,1789.75,0.10,146394.44,787.24,1005.18,1,63.95,3.759,131.376667,6.943,B,126512,47,11,4,2012
4,S10_D72,2012-11-30,NaN,0,4304.00,0.00,8935.00,175.68,12676.61,1,64.13,3.719,131.427533,6.943,B,126512,48,11,4,2012


## 7b. Fix Missing CPI / Unemployment In The Test Frame

Kaggle's `features.csv` only reports CPI and Unemployment for the first ~26 weeks of the test horizon — the last ~13 weeks per store are NaN. Forward-filling the last known value within each store is a defensible imputation that lets exogenous-aware models (AutoARIMA, LightGBM, AutoNBEATS, AutoNHITS) consume these columns without breaking.

We fill within `Store` (not within `unique_id`) because CPI and Unemployment are store-level — every department at the same store sees the same value.

In [26]:
# Recover the original Store number from unique_id so we can group by it.
# (We dropped Store as a column when we standardized to the Nixtla schema.)
test_top["Store"] = test_top["unique_id"].str.extract(r"S(\d+)_D")[0].astype(int)

# Forward-fill CPI and Unemployment within each store, in time order.
test_top = (
    test_top
    .sort_values(["Store", "ds"])
    .assign(
        CPI          = lambda d: d.groupby("Store")["CPI"].ffill(),
        Unemployment = lambda d: d.groupby("Store")["Unemployment"].ffill(),
    )
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
    .drop(columns=["Store"])  # don't need it in the final saved frame
)

# Confirm the fix worked.
print("CPI NaN remaining         :", test_top["CPI"].isna().sum())
print("Unemployment NaN remaining:", test_top["Unemployment"].isna().sum())

CPI NaN remaining         : 0
Unemployment NaN remaining: 0


## Notebook 01 — Done

**Outputs:**
- `..\data\walmart_top20_train.parquet` — top-20 Store/Dept series with covariates & calendar features
- `..\data\walmart_top20_test.parquet` — same schema, Kaggle test dates, `y` = NaN

**Next:** Notebook 02 will load the train parquet and run Naive, SeasonalNaive, AutoETS, and AutoARIMA with 5-fold non-overlapping CV at `h = 4` and `season_length = 52`.


In [ ]:
df = (
    train
    # Nixtla naming conventions: unique_id identifies the series, ds is timestamp, y is target.
    .rename(columns={"Date": "ds", "Weekly_Sales": "y"})
    .assign(unique_id = lambda x: "S" + x["Store"].astype(str) + "_D" + x["Dept"].astype(str))
    # Attach the features.csv covariates (weekly, per store).
    .merge(
        features.rename(columns={"Date": "ds"}),
        on=["Store", "ds"],
        how="left",
    )
    # Attach static store metadata (Type and Size — constant per store).
    .merge(
        stores.rename(columns={"Type": "store_type", "Size": "store_size"}),
        on="Store",
        how="left",
    )
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

print("Merged training frame shape:", df.shape)
df.head()

Merged training frame shape: (421570, 18)


,Store,Dept,ds,y,IsHoliday,unique_id,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,had_markdown,store_type,store_size
0,10,1,2010-02-05,40212.84,False,S10_D1,54.34,2.962,0.0,0.0,0.0,0.0,0.0,126.442065,9.765,0,B,126512
1,10,1,2010-02-12,67699.32,True,S10_D1,49.96,2.828,0.0,0.0,0.0,0.0,0.0,126.496258,9.765,0,B,126512
2,10,1,2010-02-19,49748.33,False,S10_D1,58.22,2.915,0.0,0.0,0.0,0.0,0.0,126.526286,9.765,0,B,126512
3,10,1,2010-02-26,33601.22,False,S10_D1,52.77,2.825,0.0,0.0,0.0,0.0,0.0,126.552286,9.765,0,B,126512
4,10,1,2010-03-05,36572.44,False,S10_D1,55.92,2.877,0.0,0.0,0.0,0.0,0.0,126.578286,9.765,0,B,126512
